In [1]:
# Instalación de librerías (descomenta la línea si no las tienes instaladas)
# !pip install gymnasium numpy matplotlib

import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

# Creamos el entorno del Taxi
env = gym.make("Taxi-v3", render_mode="ansi")

# Extraemos las dimensiones para nuestra matriz
estados = env.observation_space.n
acciones = env.action_space.n

print(f"Número de Estados: {estados}")
print(f"Número de Acciones: {acciones}")

Número de Estados: 500
Número de Acciones: 6


In [2]:
# Inicializamos la Q-Table con ceros
q_table = np.zeros([estados, acciones])

print("Forma de la Q-Table:", q_table.shape)

Forma de la Q-Table: (500, 6)


In [3]:
# Hiperparámetros de Aprendizaje
alpha = 0.1    # Tasa de aprendizaje (Cuánto confía en la nueva información vs la antigua)
gamma = 0.6    # Factor de descuento (Importancia de las recompensas futuras vs inmediatas)

# Hiperparámetros de Exploración (Epsilon-Greedy)
epsilon = 0.1  # Probabilidad del 10% de tomar una acción aleatoria para explorar el mapa

# Configuración del entrenamiento
episodios = 10000 # Número de partidas que jugará para aprender

In [4]:
from IPython.display import clear_output

# Listas para guardar el histórico y poder dibujar gráficas después
recompensas_por_episodio = []
penalizaciones_por_episodio = []

# Bucle principal de entrenamiento (10,000 partidas)
for i in range(1, episodios + 1):
    
    # 1. Reiniciar el entorno para una nueva partida
    estado, info = env.reset()
    
    recompensa_total = 0
    penalizaciones = 0
    terminado = False
    truncado = False
    
    # Bucle de la partida actual (hasta que el taxi deje al pasajero o haya un límite de tiempo)
    while not (terminado or truncado):
        
        # 2. Estrategia Epsilon-Greedy: ¿Explorar o Explotar?
        if random.uniform(0, 1) < epsilon:
            # Exploración: Tomar una acción 100% aleatoria para descubrir el mapa
            accion = env.action_space.sample() 
        else:
            # Explotación: Consultar la matriz y elegir la mejor acción conocida para este estado
            accion = np.argmax(q_table[estado]) 
            
        # 3. El agente ejecuta la acción y el entorno devuelve las consecuencias
        siguiente_estado, recompensa, terminado, truncado, info = env.step(accion)
        
        # Contabilizamos si ha cometido un error grave (intentar recoger/dejar donde no toca)
        if recompensa == -10:
            penalizaciones += 1
            
        # 4. LA ECUACIÓN DE BELLMAN (Actualización de la Q-Table)
        # Recuperamos lo que creíamos saber de este estado
        valor_antiguo = q_table[estado, accion]
        
        # Miramos un paso hacia el futuro: ¿cuál es el mejor valor del siguiente estado al que hemos llegado?
        proximo_maximo = np.max(q_table[siguiente_estado])
        
        # Calculamos el nuevo valor (Temporal Difference Learning)
        nuevo_valor = valor_antiguo + alpha * (recompensa + gamma * proximo_maximo - valor_antiguo)
        
        # Actualizamos la memoria del agente (la matriz)
        q_table[estado, accion] = nuevo_valor
        
        # El agente avanza físicamente al siguiente estado
        estado = siguiente_estado
        recompensa_total += recompensa
        
    # Guardamos los resultados de la partida para las estadísticas
    recompensas_por_episodio.append(recompensa_total)
    penalizaciones_por_episodio.append(penalizaciones)
    
    # Limpiamos la consola y mostramos el progreso cada 1,000 partidas
    if i % 1000 == 0:
        clear_output(wait=True)
        print(f"Entrenando... Episodio: {i} / {episodios}")

print("¡Entrenamiento completado!")

Entrenando... Episodio: 10000 / 10000
¡Entrenamiento completado!


In [ ]:
import matplotlib.pyplot as plt

# Función para suavizar las gráficas (Media móvil)
def moving_average(data, window_size=100):
    return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

# Configuración de la figura
plt.figure(figsize=(15, 5))

# Gráfica 1: Recompensas totales
plt.subplot(1, 2, 1)
plt.plot(moving_average(recompensas_por_episodio))
plt.title('Evolución de la Recompensa (Suavizada)')
plt.xlabel('Episodio')
plt.ylabel('Recompensa Promedio')

# Gráfica 2: Penalizaciones (Errores)
plt.subplot(1, 2, 2)
plt.plot(moving_average(penalizaciones_por_episodio))
plt.title('Evolución de las Penalizaciones (Suavizada)')
plt.xlabel('Episodio')
plt.ylabel('Media de Errores (-10 pts)')

plt.tight_layout()
plt.show()